# Relatório de Pré-Decolagem AuroraSiger com Apoio de IA

Este notebook implementa:

- verificação de segurança por **regras fixas**;
- apoio da **IA** com estimativa de risco;
- detecção de **anomalias**;
- análise energética da decolagem.

A decisão final de **PRONTO PARA DECOLAR** ou **DECOLAGEM ABORTADA** é feita pelas regras de segurança.  
A IA atua apenas como análise de apoio.

## Bibliotecas utilizadas

Neste projeto, utilizamos:

- **`pandas`**: para ler e organizar o dataset sintético em formato CSV;
- **`DecisionTreeClassifier`**: para estimar o nível de risco da operação;
- **`IsolationForest`**: para detectar possíveis anomalias nos dados.

As bibliotecas foram escolhidas por permitirem integrar análise de dados e inteligência artificial de forma prática e organizada.

In [12]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import IsolationForest

## Dados sintéticos utilizados

Neste projeto, utilizamos um **dataset sintético de telemetria** armazenado no arquivo `telemetria_sintetica.csv`, que foi gerado por Inteligência Artificial.

Abaixo, apresentamos uma prévia das primeiras linhas do dataset utilizado no experimento.

In [13]:
CAMINHO_CSV = "telemetria_sintetica.csv"
df = pd.read_csv(CAMINHO_CSV)
display(df.head())


,timestamp,internal_temp_c,external_temp_c,battery_voltage_v,battery_current_a,battery_soc_percent,battery_capacity_ah,energy_available_kwh,power_load_kw,energy_loss_percent,tank_pressure_bar,structural_integrity,critical_modules_status,telemetry_link_status,estimated_autonomy_min,launch_decision
0,2026-03-18T00:00:00,28.87,-4.12,47.65,42.32,89.46,107.07,4.564,22.84,2.52,116.10,1,1,1,45,ABORT
1,2026-03-18T00:01:00,18.51,2.65,49.03,22.65,67.95,106.00,3.531,15.90,3.32,124.46,1,1,1,45,ABORT
2,2026-03-18T00:02:00,31.76,-4.77,50.83,89.81,73.61,86.22,3.226,24.14,4.02,99.64,1,1,1,45,ABORT
3,2026-03-18T00:03:00,19.64,24.66,49.62,100.71,89.19,101.45,4.490,24.46,4.27,122.60,1,1,1,45,ABORT
4,2026-03-18T00:04:00,32.10,16.65,51.17,77.74,88.18,81.83,3.692,9.56,3.74,98.99,1,1,1,45,ABORT


## Funções principais

Neste bloco, definimos:

- a conversão de uma linha do dataset para o formato de telemetria;
- a verificação da decolagem por regras;
- a classificação do risco com apoio da IA;
- o cálculo da análise energética.

In [14]:


def carregar_dataset(caminho_csv):
    return pd.read_csv(caminho_csv)


def linha_para_telemetria(row):
    return {
        "timestamp": row["timestamp"],
        "temperatura_interna": row["internal_temp_c"],
        "temperatura_externa": row["external_temp_c"],
        "tensao_bateria": row["battery_voltage_v"],
        "corrente_bateria": row["battery_current_a"],
        "nivel_energia": row["battery_soc_percent"],
        "capacidade_bateria_ah": row["battery_capacity_ah"],
        "energia_disponivel_kwh": row["energy_available_kwh"],
        "carga_kw": row["power_load_kw"],
        "perdas_energeticas": row["energy_loss_percent"],
        "pressao_tanques": row["tank_pressure_bar"],
        "integridade_estrutural": row["structural_integrity"],
        "status_modulos_criticos": row["critical_modules_status"],
        "status_link_telemetria": row["telemetry_link_status"],
        "autonomia_estimada_dataset_min": row["estimated_autonomy_min"],
        "decisao_dataset": row["launch_decision"],
    }


def verificar_decolagem(dados):
    falhas = []

    if not (20 <= dados["temperatura_interna"] <= 32):
        falhas.append("Temperatura interna fora da faixa segura.")

    if not (0 <= dados["temperatura_externa"] <= 28):
        falhas.append("Temperatura externa fora da faixa segura.")

    if not (47 <= dados["tensao_bateria"] <= 51.5):
        falhas.append("Tensão da bateria fora da faixa segura.")

    if not (25 <= dados["corrente_bateria"] <= 100):
        falhas.append("Corrente da bateria fora da faixa segura.")

    if dados["nivel_energia"] < 70:
        falhas.append("Nível de energia insuficiente.")

    if dados["perdas_energeticas"] > 7:
        falhas.append("Perdas energéticas acima do limite.")

    if not (100 <= dados["pressao_tanques"] <= 140):
        falhas.append("Pressão dos tanques fora da faixa segura.")

    if dados["integridade_estrutural"] != 1:
        falhas.append("Integridade estrutural comprometida.")

    if dados["status_modulos_criticos"] != 1:
        falhas.append("Falha nos módulos críticos.")

    if dados["status_link_telemetria"] != 1:
        falhas.append("Falha no link de telemetria.")

    if len(falhas) == 0:
        return "PRONTO PARA DECOLAR", falhas

    return "DECOLAGEM ABORTADA", falhas


def classificar_risco_por_probabilidade(prob_ready):
    if prob_ready >= 0.80:
        return "RISCO_BAIXO"
    elif prob_ready >= 0.50:
        return "RISCO_MODERADO"
    return "RISCO_ALTO"


def calcular_analise_energetica(energia_disponivel_kwh, perdas_percentual, carga_kw):
    energia_util = energia_disponivel_kwh * (1 - perdas_percentual / 100)

    consumo_estimado_decolagem_kwh = carga_kw / 60

    energia_restante_apos_decolagem = max(
        energia_util - consumo_estimado_decolagem_kwh, 0
    )

    autonomia_estimada_min = (energia_util / carga_kw) * 60 if carga_kw > 0 else 0

    return (
        energia_util,
        consumo_estimado_decolagem_kwh,
        energia_restante_apos_decolagem,
        autonomia_estimada_min,
    )

## Treinamento dos modelos de IA

Aqui, carregamos o dataset sintético e treinamos:

- um modelo para **estimativa de risco**, com base na decisão `READY` ou `ABORT`;
- um modelo para **detecção de anomalias**.

A IA não toma a decisão final da decolagem.  
Ela complementa a análise feita pelas regras do sistema.

In [15]:
CAMINHO_CSV = "telemetria_sintetica.csv"

df = carregar_dataset(CAMINHO_CSV)

colunas_features = [
    "internal_temp_c",
    "external_temp_c",
    "battery_voltage_v",
    "battery_current_a",
    "battery_soc_percent",
    "energy_available_kwh",
    "power_load_kw",
    "energy_loss_percent",
    "tank_pressure_bar",
    "estimated_autonomy_min",
]

modelo_decisao = DecisionTreeClassifier(random_state=42)
modelo_decisao.fit(df[colunas_features], df["launch_decision"])

modelo_anomalia = IsolationForest(random_state=42, contamination=0.1)
modelo_anomalia.fit(df[colunas_features])

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.1
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


## Simulação da telemetria

Nessa etapa, utilizamos uma linha do próprio dataset como se fosse a leitura atual dos sensores da nave.

Por padrão, o notebook seleciona a primeira linha cuja decisão no dataset seja `READY`.  
Caso não exista nenhuma, utiliza a primeira linha disponível.

In [16]:
indices_ready = df.index[df["launch_decision"] == "READY"].tolist()
INDICE_AMOSTRA = indices_ready[0] if indices_ready else 0

row_atual = df.iloc[INDICE_AMOSTRA]
telemetria = linha_para_telemetria(row_atual)
features_atual = df.loc[[INDICE_AMOSTRA], colunas_features]

display(pd.DataFrame([telemetria]))

,timestamp,temperatura_interna,temperatura_externa,tensao_bateria,corrente_bateria,nivel_energia,capacidade_bateria_ah,energia_disponivel_kwh,carga_kw,perdas_energeticas,pressao_tanques,integridade_estrutural,status_modulos_criticos,status_link_telemetria,autonomia_estimada_dataset_min,decisao_dataset
0,2026-03-18T00:25:00,27.93,12.6,51.12,35.74,98.43,83.2,4.186,8.72,5.57,128.76,1,1,1,45,READY


## Execução da verificação

Aqui o sistema:

1. avalia as regras de segurança;
2. verifica com a IA o nível de risco;
3. identifica se há anomalias;
4. imprime o relatório de pré-decolagem.

In [17]:
status, falhas = verificar_decolagem(telemetria)

probs = modelo_decisao.predict_proba(features_atual)[0]
classes = list(modelo_decisao.classes_)

if "READY" in classes:
    prob_ready = probs[classes.index("READY")]
else:
    prob_ready = 0.0

risco_ia = classificar_risco_por_probabilidade(prob_ready)

anomalia = modelo_anomalia.predict(features_atual)[0]

if anomalia == -1:
    mensagem_ia = "comportamento anômalo detectado."
else:
    mensagem_ia = "nenhuma anomalia detectada."

print("=== RELATÓRIO DE PRÉ-DECOLAGEM ===")
for chave, valor in telemetria.items():
    print(f"{chave}: {valor}")

print("\nResultado final:", status)
print("Nível de risco estimado pela IA:", risco_ia)
print("IA:", mensagem_ia)

if falhas:
    print("Motivos da abortagem:")
    for falha in falhas:
        print("-", falha)

=== RELATÓRIO DE PRÉ-DECOLAGEM ===
timestamp: 2026-03-18T00:25:00
temperatura_interna: 27.93
temperatura_externa: 12.6
tensao_bateria: 51.12
corrente_bateria: 35.74
nivel_energia: 98.43
capacidade_bateria_ah: 83.2
energia_disponivel_kwh: 4.186
carga_kw: 8.72
perdas_energeticas: 5.57
pressao_tanques: 128.76
integridade_estrutural: 1
status_modulos_criticos: 1
status_link_telemetria: 1
autonomia_estimada_dataset_min: 45
decisao_dataset: READY

Resultado final: PRONTO PARA DECOLAR
Nível de risco estimado pela IA: RISCO_BAIXO
IA: nenhuma anomalia detectada.


## Análise energética

Agora calculamos:

- energia disponível antes das perdas;
- energia útil após as perdas;
- consumo estimado da decolagem;
- energia restante após a decolagem;
- autonomia estimada.

Nesta etapa, os valores são obtidos a partir do próprio dataset sintético.

In [18]:
energia_util, consumo_estimado_decolagem_kwh, energia_restante_apos_decolagem, autonomia_estimada_min = calcular_analise_energetica(
    telemetria["energia_disponivel_kwh"],
    telemetria["perdas_energeticas"],
    telemetria["carga_kw"],
)

print("=== ANÁLISE ENERGÉTICA ===")
print(f"Energia disponível antes das perdas: {telemetria['energia_disponivel_kwh']:.3f} kWh")
print(f"Energia útil após perdas: {energia_util:.3f} kWh")
print(f"Consumo estimado da decolagem (1 min): {consumo_estimado_decolagem_kwh:.3f} kWh")
print(f"Energia restante após decolagem: {energia_restante_apos_decolagem:.3f} kWh")
print(f"Autonomia estimada após perdas: {autonomia_estimada_min:.2f} min")
print(f"Autonomia registrada no dataset: {telemetria['autonomia_estimada_dataset_min']} min")

=== ANÁLISE ENERGÉTICA ===
Energia disponível antes das perdas: 4.186 kWh
Energia útil após perdas: 3.953 kWh
Consumo estimado da decolagem (1 min): 0.145 kWh
Energia restante após decolagem: 3.808 kWh
Autonomia estimada após perdas: 27.20 min
Autonomia registrada no dataset: 45 min


## Conclusão

O sistema combina duas abordagens:

- **regras fixas:** responsáveis pela decisão final de segurança;
- **IA:** utilizada como apoio para estimativa de risco e detecção de anomalias.

Essa separação continua sendo importante porque, em sistemas críticos, a decisão operacional deve ser baseada em critérios explícitos e auditáveis, enquanto a IA deve atuar como ferramenta complementar de análise.